## C . Spark : Parallelisation of the image processing algorithm Median Filter

-> **Expected Results:**

    - Spark parallel median_filter.py
    
    - Execute on lena_noizy.jpg and generate lena_filter.jpg

In [1]:
import pyspark
from pyspark.sql import SparkSession
import imageio.v2 as imageio
import os
import numpy as np

In [2]:
def readImg(path):
    img = imageio.imread(path)
    im = np.array(img, dtype='uint8')
    return im

def writeImg(path, buf):
    imageio.imwrite(path, buf)

def part_median_filter(local_data, nb_partitions):
    part_id = local_data[0]
    first = local_data[1]
    end = local_data[2]
    buf = local_data[3]
    
    # Extract the part of the image with overlap
    part_buf = buf[first:end, :, :]
    height, width, channels = part_buf.shape
    
    # Create an empty buffer for the filtered part
    filtered_part = np.zeros((height, width, channels), dtype='uint8')
    
    # Apply median filter to each channel separately
    for c in range(channels):
        for i in range(1, height - 1):
            for j in range(1, width - 1):
                # Extract the 3x3 neighborhood for the current pixel
                neighborhood = part_buf[i-1:i+2, j-1:j+2, c]
                # Compute the median and assign it to the filtered pixel
                filtered_part[i, j, c] = np.median(neighborhood)
    
    # Remove the overlap before returning the result
    if part_id == 0:
        return part_id, filtered_part[:-1, :, :]  # Remove bottom overlap
    elif part_id == nb_partitions - 1:
        return part_id, filtered_part[1:, :, :]   # Remove top overlap
    else:
        return part_id, filtered_part[1:-1, :, :] # Remove top and bottom overlap


In [3]:
def main():
    data_dir = './data'
    file = os.path.join(data_dir, 'lena_noisy.jpg')
    img_buf = readImg(file)
    print('SHAPE', img_buf.shape)
    print('IMG\n', img_buf)
    nx = img_buf.shape[0]
    ny = img_buf.shape[1]
    
    ###########################################################################
    #
    # SPLIT IMAGES INTO NB_PARTITIONS PARTS WITH OVERLAP
    nb_partitions = 8
    print("NB PARTITIONS : ", nb_partitions)
    data = []
    begin = 0
    block_size = nx // nb_partitions
    for ip in range(nb_partitions):
        end = min(begin + block_size + 2, nx)  # Overlap by 2 pixels
        data.append([ip, begin, end, img_buf])
        begin = end - 2  # Overlap by 2 pixels
    
    ###########################################################################
    #
    # CREATE SPARKSESSION
    spark = SparkSession.builder \
                .appName("MedianFilter") \
                    .config("spark.driver.host", "localhost").config("spark.driver.memory", "2g").config("spark.executor.memory", "2g")\
                        .getOrCreate () 
    
    ###########################################################################
    #
    # PARALLEL MEDIAN FILTER COMPUTATION
    data_rdd = spark.sparkContext.parallelize(data, nb_partitions)
    
    # Pass nb_partitions as an additional argument to part_median_filter
    result_rdd = data_rdd.map(lambda x: part_median_filter(x, nb_partitions))
    result_data = result_rdd.collect()
    
    # Reconstruct the image from the filtered parts
    new_img_buf = np.zeros((nx, ny, 3), dtype='uint8')
    for part_id, filtered_part in result_data:
        first = data[part_id][1]
        end = data[part_id][2]
        if part_id == 0:
            new_img_buf[first:end-1, :, :] = filtered_part  # Remove bottom overlap
        elif part_id == nb_partitions - 1:
            new_img_buf[first+1:end, :, :] = filtered_part  # Remove top overlap
        else:
            new_img_buf[first+1:end-1, :, :] = filtered_part  # Remove top and bottom overlap
    
    print('CREATE NEW PICTURE FILE')
    filter_file = os.path.join(data_dir, 'lena_filter.jpg')
    writeImg(filter_file, new_img_buf)
    
    ###########################################################################
    #
    # STOP SPARKSESSION
    spark.stop()

if __name__ == '__main__':
    main()

SHAPE (128, 128, 3)
IMG
 [[[233 159 122]
  [228 159 118]
  [223 165 115]
  ...
  [181 116  98]
  [231 165 153]
  [223 156 147]]

 [[220 155 127]
  [226 166 132]
  [232 179 139]
  ...
  [199 134 116]
  [205 139 125]
  [162  98  86]]

 [[207 161 148]
  [206 160 144]
  [254 209 188]
  ...
  [156  93  75]
  [134  71  56]
  [114  52  37]]

 ...

 [[ 88  52  56]
  [155 119 123]
  [109  68  76]
  ...
  [131  78 108]
  [104  55  74]
  [107  61  72]]

 [[ 93  62  70]
  [219 186 193]
  [ 95  56  59]
  ...
  [100  56  71]
  [118  82  96]
  [122  90 105]]

 [[ 89  62  71]
  [ 81  50  56]
  [ 86  47  50]
  ...
  [109  69  77]
  [125  94 109]
  [214 190 206]]]
NB PARTITIONS :  8
CREATE NEW PICTURE FILE
